# Order Event Producer

Reads real orders from `orders.csv` and `order_items.csv`, joins and groups them into order events, and streams them into Event Hub (`evh_brazilian_ecommerce`) one at a time — simulating customers placing orders in real time.

**Run this while `02_stream_orders_to_bronze` is actively running.** The consumer uses `startingOffsets: latest`, so it only picks up events sent after it started listening.

## 1. Install the Event Hub SDK

In [0]:
%pip install azure-eventhub

In [0]:
%restart_python

## 2. Imports

In [0]:
import json
import random
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from azure.eventhub import EventHubProducerClient, EventData

## 3. Parameters
- `num_events` — how many real orders to pull and send this run
- `sleep_seconds` — pause between sends, to simulate a steady stream rather than a burst
- `include_discount_code` — the schema-evolution toggle: off sends the original order shape, on adds `discount_code` to every event from that point on

In [0]:
dbutils.widgets.text("num_events", "20", "Number of Events")
dbutils.widgets.text("sleep_seconds", "1", "Seconds Between Events")
dbutils.widgets.dropdown("include_discount_code", "false", ["false", "true"], "Include discount_code")

num_events = int(dbutils.widgets.get("num_events"))
sleep_seconds = float(dbutils.widgets.get("sleep_seconds"))
include_discount_code = dbutils.widgets.get("include_discount_code") == "true"

## 4. Event Hub connection
Connection string comes from the governed, Key Vault-backed secret scop, never hardcoded, never printed.

In [0]:
connection_string = dbutils.secrets.get(
    scope="e-commerce-bronze-scope",
    key="evh-brazilian-ecommerce"
)
eventhub_name = "evh_brazilian_ecommerce"

## 5. Build order events from real data
`orders.csv` has one row per order with the real purchase timestamp. `order_items.csv` has one row per item on that order, so the same `product_id` can appear more than once under one `order_id`. Grouping by `order_id` + `product_id` turns repeated rows into a `quantity` (row count) and a summed `price`.

`DISCOUNT_CODES` isn't real Olist data — there's no discount field in the dataset, which is exactly why it's a good schema-evolution example: it's genuinely new information the source never had.

Orders are pulled in random order (`F.rand()`) rather than chronological order, so repeated demo runs don't always send the same first 20 orders.

In [0]:
DISCOUNT_CODES = ["SUMMER25", "WELCOME10", "FLASH15", None]

orders_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/dbr_dev/brazilian_ecommerce_bronze/landing/orders/orders.csv")
    .select(
        "order_id",
        "customer_id",
        F.to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm").alias("order_purchase_timestamp"),
    )
)

order_items_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/dbr_dev/brazilian_ecommerce_bronze/landing/order_items/order_items.csv")
    .select("order_id", "product_id", "price")
)

order_events_df = (
    orders_df
    .join(order_items_df, on="order_id", how="inner")
    .groupBy("order_id", "customer_id", "product_id", "order_purchase_timestamp")
    .agg(
        F.count("*").alias("quantity"),
        F.round(F.sum("price"), 2).alias("price"),
    )
    .orderBy("order_purchase_timestamp")
    .orderBy(F.rand())          # randomize orders, so we will receive random orders instead of first 20 each time
    .limit(num_events)
)

order_events = order_events_df.collect()
print(f"Loaded {len(order_events)} order events, ready to send")

## 6. Send events to Event Hub
One event per order, JSON-encoded, sent with a short pause between each to simulate a live stream.

In [0]:
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_string,
    eventhub_name=eventhub_name,
)

sent_count = 0
try:
    for row in order_events:
        order = {
            "order_id": row["order_id"],
            "customer_id": row["customer_id"],
            "product_id": row["product_id"],
            "quantity": row["quantity"],
            "price": row["price"],
            "order_timestamp": row["order_purchase_timestamp"].strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
        if include_discount_code:
            order["discount_code"] = random.choice(DISCOUNT_CODES)

        batch = producer.create_batch()
        batch.add(EventData(json.dumps(order)))
        producer.send_batch(batch)
        sent_count += 1
        print(f"Sent order {order['order_id']} (product {order['product_id']}, qty {order['quantity']})")
        time.sleep(sleep_seconds)
finally:
    producer.close()

print(f"Done — sent {sent_count} real order events (discount_code included: {include_discount_code})")